## Homework 4

In [3]:
import pandas as pd

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import GradientBoostingClassifier

In [4]:

train = pd.read_csv("playground-series-s6e4/train.csv")
train.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [5]:
train.drop(columns=["id"], inplace=True)

In [6]:
from sklearn.preprocessing import PolynomialFeatures

# work on a copy of train data
train_fe = train.copy()

# separate target and numeric predictors
target_col = "Irrigation_Need"
num_cols = train_fe.drop(columns=[target_col]).select_dtypes(include=["number"]).columns.tolist()

# create all pairwise interactions + squared terms (degree=2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(train_fe[num_cols])
poly_feature_names = poly.get_feature_names_out(num_cols)

# build engineered feature dataframe and add target back
train_poly = pd.DataFrame(X_poly, columns=poly_feature_names, index=train_fe.index)
train_poly[target_col] = train_fe[target_col]

print("Original numeric feature count:", len(num_cols))
print("Engineered feature count:", len(poly_feature_names))
train_poly.head()

Original numeric feature count: 11
Engineered feature count: 77


,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Field_Area_hectare,...,Sunlight_Hours Wind_Speed_kmh,Sunlight_Hours Field_Area_hectare,Sunlight_Hours Previous_Irrigation_mm,Wind_Speed_kmh^2,Wind_Speed_kmh Field_Area_hectare,Wind_Speed_kmh Previous_Irrigation_mm,Field_Area_hectare^2,Field_Area_hectare Previous_Irrigation_mm,Previous_Irrigation_mm^2,Irrigation_Need
0,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,0.82,...,99.0610,4.8380,661.7440,281.9041,13.7678,1883.1664,0.6724,91.9712,12579.8656,Low
1,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,5.27,...,23.6622,36.7846,329.1768,11.4921,17.8653,159.8724,27.7729,248.5332,2224.0656,Low
2,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,8.24,...,23.2925,49.8520,667.7990,14.8225,31.7240,424.9630,67.8976,909.5312,12183.7444,Low
3,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,8.32,...,21.0672,75.8784,491.1120,5.3361,19.2192,124.3935,69.2224,448.0320,2899.8225,Medium
4,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,7.37,...,96.8830,51.2215,647.6705,194.3236,102.7378,1299.0686,54.3169,686.8103,8684.3761,Low


In [7]:
# leakage-safe CV target encoding for categoricals + polynomial numeric features
from sklearn.model_selection import StratifiedKFold

cat_cols = train_fe.drop(columns=[target_col]).select_dtypes(include=["object", "category"]).columns.tolist()
y_all_models = train_fe[target_col].copy()

# out-of-fold encoded categorical frame (dense, not sparse)
X_cat_cv_encoded = pd.DataFrame(index=train_fe.index)
y_onehot = pd.get_dummies(y_all_models, prefix="target")
class_cols = y_onehot.columns.tolist()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for col in cat_cols:
    # initialize encoded columns for this categorical feature
    for cls in class_cols:
        X_cat_cv_encoded[f"{col}__{cls}_te"] = 0.0

    # build out-of-fold encodings to avoid leakage
    for tr_idx, val_idx in skf.split(train_fe, y_all_models):
        X_tr_col = train_fe.iloc[tr_idx][col]
        y_tr_fold = y_onehot.iloc[tr_idx]

        fold_stats = y_tr_fold.groupby(X_tr_col).mean()
        fold_prior = y_tr_fold.mean()

        val_map = train_fe.iloc[val_idx][col].map(fold_stats.to_dict("index"))

        for cls in class_cols:
            X_cat_cv_encoded.iloc[val_idx, X_cat_cv_encoded.columns.get_loc(f"{col}__{cls}_te")] = [
                x.get(cls, fold_prior[cls]) if isinstance(x, dict) else fold_prior[cls]
                for x in val_map
            ]

# combine with numeric polynomial features
X_num_poly = train_poly.drop(columns=[target_col])
X_all_models = pd.concat([X_num_poly, X_cat_cv_encoded], axis=1)

train_all_models = X_all_models.copy()
train_all_models[target_col] = y_all_models

print("Polynomial numeric features:", X_num_poly.shape[1])
print("CV-encoded categorical features:", X_cat_cv_encoded.shape[1])
print("Total features for shared dataset:", X_all_models.shape[1])
train_all_models.head()

Polynomial numeric features: 77
CV-encoded categorical features: 24
Total features for shared dataset: 101


,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Field_Area_hectare,...,Water_Source__target_High_te,Water_Source__target_Low_te,Water_Source__target_Medium_te,Mulching_Used__target_High_te,Mulching_Used__target_Low_te,Mulching_Used__target_Medium_te,Region__target_High_te,Region__target_Low_te,Region__target_Medium_te,Irrigation_Need
0,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,0.82,...,0.033886,0.604026,0.362088,0.058510,0.445521,0.495969,0.028620,0.594028,0.377352,Low
1,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,5.27,...,0.043382,0.582710,0.373908,0.007955,0.730571,0.261475,0.034342,0.586119,0.379540,Low
2,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,8.24,...,0.032654,0.562572,0.404774,0.007955,0.730571,0.261475,0.034124,0.576139,0.389736,Low
3,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,8.32,...,0.043642,0.583571,0.372786,0.008041,0.730450,0.261509,0.035079,0.586676,0.378246,Medium
4,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,7.37,...,0.043401,0.583958,0.372642,0.058535,0.444845,0.496620,0.034795,0.586650,0.378555,Low


In [8]:
# separate training data for CatBoost using raw categorical features
X_catboost_raw = train_fe.drop(columns=[target_col]).copy()
y_catboost_raw = train_fe[target_col].copy()

cat_features = X_catboost_raw.select_dtypes(include=["object", "category"]).columns.tolist()

print("CatBoost raw feature shape:", X_catboost_raw.shape)
print("Number of categorical features for CatBoost:", len(cat_features))
print("Categorical features:", cat_features)
X_catboost_raw.head()

CatBoost raw feature shape: (630000, 19)
Number of categorical features for CatBoost: 8
Categorical features: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South


In [10]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

# use a stratified sample for faster debugging
sample_n = 50_000
sample_n = min(sample_n, len(y_all_models))

# one stratified index so GBM/LGBM rows match CatBoost rows row-for-row
sample_idx = y_all_models.groupby(y_all_models, group_keys=False).apply(
    lambda s: s.sample(max(1, int(len(s) * sample_n / len(y_all_models))), random_state=42)
).index

X_all_models_s = X_all_models.loc[sample_idx].reset_index(drop=True)
y_all_models_s = y_all_models.loc[sample_idx].reset_index(drop=True)

X_catboost_raw_s = X_catboost_raw.loc[sample_idx].reset_index(drop=True)
y_catboost_raw_s = y_catboost_raw.loc[sample_idx].reset_index(drop=True)


In [ ]:
# 2-fold CV setup
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

# baseline models with progress verbosity
baseline_models = {
    "GradientBoosting": {
        "model": GradientBoostingClassifier(random_state=42, verbose=2),
        "X": X_all_models_s,
        "y": y_all_models_s,
        "fit_kwargs": {}
    },
    "LightGBM": {
        "model": LGBMClassifier(random_state=42, verbose=2),
        "X": X_all_models_s,
        "y": y_all_models_s,
        "fit_kwargs": {}
    },
    "CatBoost": {
        "model": CatBoostClassifier(random_state=42, verbose=2),
        "X": X_catboost_raw_s,
        "y": y_catboost_raw_s,
        "fit_kwargs": {"cat_features": cat_features}
    }
}

# evaluate each model with balanced accuracy
cv_results = {}

for name, cfg in baseline_models.items():
    print(f"\nRunning {name} on {len(cfg['y'])} sampled rows...")
    X_data, y_data = cfg["X"], cfg["y"]
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X_data, y_data), start=1):
        print(f"{name} | Fold {fold}/2")
        X_tr, X_val = X_data.iloc[tr_idx], X_data.iloc[val_idx]
        y_tr, y_val = y_data.iloc[tr_idx], y_data.iloc[val_idx]

        model = clone(cfg["model"])
        model.fit(X_tr, y_tr, **cfg["fit_kwargs"])
        preds = model.predict(X_val)

        score = balanced_accuracy_score(y_val, preds)
        fold_scores.append(score)
        print(f"{name} | Fold {fold} balanced_accuracy={score:.5f}")

    cv_results[name] = fold_scores

# print fold-by-fold and summary
for name, scores in cv_results.items():
    print(f"\n{name} balanced accuracy (2-fold, sampled):")
    print([round(s, 5) for s in scores])
    print(f"mean={sum(scores)/len(scores):.5f}, std={pd.Series(scores).std():.5f}")
    print("-" * 60)

In [15]:
import numpy as np
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

# (1) smaller stratified tuning subset (2) fewer trials (5) single holdout vs CV — faster Optuna
TUNE_SAMPLE_N = 20_000
N_TRIALS = 15
# sklearn GradientBoosting is slow: smaller data + random sampler + fewer trials only for GBM
GBM_TUNE_N = 5_000
N_TRIALS_GBM = 10

# aligned rows across X_all_models_s and X_catboost_raw_s (same indices as baseline sample)
_tune_size = min(TUNE_SAMPLE_N, len(y_all_models_s))
_tune_idx, _ = train_test_split(
    np.arange(len(y_all_models_s)),
    train_size=_tune_size,
    stratify=y_all_models_s,
    random_state=1337,
    shuffle=True,
)
X_tune_shared = X_all_models_s.iloc[_tune_idx].reset_index(drop=True)
y_tune_shared = y_all_models_s.iloc[_tune_idx].reset_index(drop=True)
X_tune_cat = X_catboost_raw_s.iloc[_tune_idx].reset_index(drop=True)
y_tune_cat = y_catboost_raw_s.iloc[_tune_idx].reset_index(drop=True)

print(
    f"Optuna: {_tune_size} stratified rows | "
    f"objective = balanced acc on one holdout (20% val, stratified)"
)

# extra-small stratified subset for sklearn GBM only (fits ~4k train per trial vs ~16k)
_gbm_n = min(GBM_TUNE_N, len(y_tune_shared))
_gbm_idx, _ = train_test_split(
    np.arange(len(y_tune_shared)),
    train_size=_gbm_n,
    stratify=y_tune_shared,
    random_state=4242,
    shuffle=True,
)
X_tune_gbm = X_tune_shared.iloc[_gbm_idx].reset_index(drop=True)
y_tune_gbm = y_tune_shared.iloc[_gbm_idx].reset_index(drop=True)
print(
    f"GBM Optuna: {_gbm_n} stratified rows (subset of tuning data), "
    f"{N_TRIALS_GBM} random trials, RandomSampler"
)

optuna.logging.set_verbosity(optuna.logging.INFO)


from sklearn.base import clone


def _holdout_balanced_accuracy(model, X, y, fit_kwargs=None, val_fraction=0.2):
    fit_kwargs = fit_kwargs or {}
    X_tr, X_val, y_tr, y_val = train_test_split(
        X, y, test_size=val_fraction, stratify=y, random_state=42, shuffle=True
    )
    m = clone(model)
    m.fit(X_tr, y_tr, **fit_kwargs)
    pred = m.predict(X_val)
    return float(balanced_accuracy_score(y_val, pred))


def _boundary_notes(best_params, grids):
    lines = []
    for name, value in best_params.items():
        if name not in grids:
            continue
        grid = grids[name]
        if value == grid[0]:
            lines.append(f"  {name}={value!r}  ← at grid MIN (try lower values)")
        elif value == grid[-1]:
            lines.append(f"  {name}={value!r}  ← at grid MAX (try higher values)")
        else:
            lines.append(f"  {name}={value!r}")
    return lines


# --- GradientBoosting (sklearn) ---
gbm_grids = {
    # shifted after best at n_estimators=100 (min), min_samples_leaf=1 (min)
    "n_estimators": [50, 100, 250, 400],
    "learning_rate": [0.03, 0.06, 0.1, 0.15],
    "max_depth": [3, 4, 5, 8],
    "min_samples_leaf": [1, 8, 30, 80],
    "subsample": [0.65, 0.85, 1.0],
}


def gbm_objective(trial):
    print(f"[GBM] Trial {trial.number} starting...", flush=True)
    params = {
        "n_estimators": trial.suggest_categorical("n_estimators", gbm_grids["n_estimators"]),
        "learning_rate": trial.suggest_categorical("learning_rate", gbm_grids["learning_rate"]),
        "max_depth": trial.suggest_categorical("max_depth", gbm_grids["max_depth"]),
        "min_samples_leaf": trial.suggest_categorical("min_samples_leaf", gbm_grids["min_samples_leaf"]),
        "subsample": trial.suggest_categorical("subsample", gbm_grids["subsample"]),
    }
    model = GradientBoostingClassifier(random_state=42, verbose=0, **params)
    score = _holdout_balanced_accuracy(model, X_tune_gbm, y_tune_gbm)
    print(f"[GBM] Trial {trial.number} done, mean_balanced_acc={score:.5f}", flush=True)
    return score


study_gbm = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.RandomSampler(seed=42),
)
study_gbm.optimize(gbm_objective, n_trials=N_TRIALS_GBM, show_progress_bar=True)

print("GradientBoosting — best holdout balanced accuracy:", round(study_gbm.best_value, 5))
print("Best params:")
for line in _boundary_notes(study_gbm.best_params, gbm_grids):
    print(line)

best_gbm = GradientBoostingClassifier(random_state=42, verbose=0, **study_gbm.best_params)

# --- LightGBM ---
lgb_grids = {
    # shifted after best at lr=max, num_leaves=min, min_child=max, subsample=min
    "n_estimators": [400, 600, 800, 1000],
    "learning_rate": [0.06, 0.1, 0.15, 0.25],
    "num_leaves": [15, 31, 63, 127],
    "min_child_samples": [50, 100, 200, 300],
    "subsample": [0.55, 0.7, 0.85],
}


def lgb_objective(trial):
    print(f"[LGBM] Trial {trial.number} starting...", flush=True)
    params = {
        "n_estimators": trial.suggest_categorical("n_estimators", lgb_grids["n_estimators"]),
        "learning_rate": trial.suggest_categorical("learning_rate", lgb_grids["learning_rate"]),
        "num_leaves": trial.suggest_categorical("num_leaves", lgb_grids["num_leaves"]),
        "min_child_samples": trial.suggest_categorical("min_child_samples", lgb_grids["min_child_samples"]),
        "subsample": trial.suggest_categorical("subsample", lgb_grids["subsample"]),
    }
    model = LGBMClassifier(random_state=42, verbosity=-1, **params)
    score = _holdout_balanced_accuracy(model, X_tune_shared, y_tune_shared)
    print(f"[LGBM] Trial {trial.number} done, mean_balanced_acc={score:.5f}", flush=True)
    return score


study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\nLightGBM — best holdout balanced accuracy:", round(study_lgb.best_value, 5))
print("Best params:")
for line in _boundary_notes(study_lgb.best_params, lgb_grids):
    print(line)

best_lgb = LGBMClassifier(random_state=42, verbosity=-1, **study_lgb.best_params)

# --- CatBoost ---
cb_grids = {
    # shifted after best at iterations=min, learning_rate=max
    "iterations": [150, 300, 600, 900],
    "learning_rate": [0.06, 0.1, 0.15, 0.25],
    "depth": [4, 6, 8, 10],
    "l2_leaf_reg": [1, 3, 10, 30],
}


def cb_objective(trial):
    print(f"[CatBoost] Trial {trial.number} starting...", flush=True)
    params = {
        "iterations": trial.suggest_categorical("iterations", cb_grids["iterations"]),
        "learning_rate": trial.suggest_categorical("learning_rate", cb_grids["learning_rate"]),
        "depth": trial.suggest_categorical("depth", cb_grids["depth"]),
        "l2_leaf_reg": trial.suggest_categorical("l2_leaf_reg", cb_grids["l2_leaf_reg"]),
    }
    model = CatBoostClassifier(random_state=42, verbose=0, **params)
    score = _holdout_balanced_accuracy(
        model, X_tune_cat, y_tune_cat, fit_kwargs={"cat_features": cat_features}
    )
    print(f"[CatBoost] Trial {trial.number} done, mean_balanced_acc={score:.5f}", flush=True)
    return score


study_cb = optuna.create_study(direction="maximize")
study_cb.optimize(cb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\nCatBoost — best holdout balanced accuracy:", round(study_cb.best_value, 5))
print("Best params:")
for line in _boundary_notes(study_cb.best_params, cb_grids):
    print(line)

best_catboost = CatBoostClassifier(
    random_state=42, verbose=0, **study_cb.best_params
)

[I 2026-04-27 16:14:34,952] A new study created in memory with name: no-name-a2971942-682b-437b-aa2b-b6868ef813ff


Optuna: 20000 stratified rows | objective = balanced acc on one holdout (20% val, stratified)
GBM Optuna: 5000 stratified rows (subset of tuning data), 10 random trials, RandomSampler


  0%|          | 0/10 [00:00<?, ?it/s]

[GBM] Trial 0 starting...
[GBM] Trial 0 done, mean_balanced_acc=0.90900
[I 2026-04-27 16:15:26,023] Trial 0 finished with value: 0.908999311923553 and parameters: {'n_estimators': 100, 'learning_rate': 0.15, 'max_depth': 8, 'min_samples_leaf': 1, 'subsample': 0.85}. Best is trial 0 with value: 0.908999311923553.
[GBM] Trial 1 starting...
[GBM] Trial 1 done, mean_balanced_acc=0.85162
[I 2026-04-27 16:15:54,467] Trial 1 finished with value: 0.8516231491585277 and parameters: {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 8, 'min_samples_leaf': 80, 'subsample': 0.65}. Best is trial 0 with value: 0.908999311923553.
[GBM] Trial 2 starting...
[GBM] Trial 2 done, mean_balanced_acc=0.88939
[I 2026-04-27 16:16:10,015] Trial 2 finished with value: 0.8893914687862982 and parameters: {'n_estimators': 50, 'learning_rate': 0.06, 'max_depth': 5, 'min_samples_leaf': 1, 'subsample': 0.85}. Best is trial 0 with value: 0.908999311923553.
[GBM] Trial 3 starting...
[GBM] Trial 3 done, mean_balanc

[I 2026-04-27 16:22:41,117] A new study created in memory with name: no-name-25a27c99-ca00-4109-9f0f-99fb13f7f1e2


...

[LGBM] Trial 0 starting...
[LGBM] Trial 0 done, mean_balanced_acc=0.94949
[I 2026-04-27 16:22:53,022] Trial 0 finished with value: 0.9494910928711572 and parameters: {'n_estimators': 1000, 'learning_rate': 0.1, 'num_leaves': 31, 'min_child_samples': 300, 'subsample': 0.85}. Best is trial 0 with value: 0.9494910928711572.
[LGBM] Trial 1 starting...
[LGBM] Trial 1 done, mean_balanced_acc=0.94957
[I 2026-04-27 16:23:01,673] Trial 1 finished with value: 0.9495687756908023 and parameters: {'n_estimators': 400, 'learning_rate': 0.15, 'num_leaves': 127, 'min_child_samples': 50, 'subsample': 0.55}. Best is trial 1 with value: 0.9495687756908023.
[LGBM] Trial 2 starting...
[LGBM] Trial 2 done, mean_balanced_acc=0.94719
[I 2026-04-27 16:23:23,003] Trial 2 finished with value: 0.9471909531002791 and parameters: {'n_estimators': 600, 'learning_rate': 0.06, 'num_leaves': 127, 'min_child_samples': 100, 'subsample': 0.55}. Best is trial 1 with value: 0.9495687756908023.
[LGBM] Trial 3 starting...
[LG

[I 2026-04-27 16:24:49,588] A new study created in memory with name: no-name-7a72c803-73c6-40d4-b281-ef12a6980ee8


[I 2026-04-27 16:24:49,579] Trial 14 finished with value: 0.9526426587808178 and parameters: {'n_estimators': 600, 'learning_rate': 0.15, 'num_leaves': 31, 'min_child_samples': 200, 'subsample': 0.55}. Best is trial 4 with value: 0.9526426587808178.

LightGBM — best holdout balanced accuracy: 0.95264
Best params:
  n_estimators=600
  learning_rate=0.15
  num_leaves=31
  min_child_samples=200
  subsample=0.55  ← at grid MIN (try lower values)


  0%|          | 0/15 [00:00<?, ?it/s]

[CatBoost] Trial 0 starting...
[CatBoost] Trial 0 done, mean_balanced_acc=0.95300
[I 2026-04-27 16:25:09,897] Trial 0 finished with value: 0.9530041503133905 and parameters: {'iterations': 900, 'learning_rate': 0.15, 'depth': 6, 'l2_leaf_reg': 3}. Best is trial 0 with value: 0.9530041503133905.
[CatBoost] Trial 1 starting...
[CatBoost] Trial 1 done, mean_balanced_acc=0.94984
[I 2026-04-27 16:25:14,508] Trial 1 finished with value: 0.9498391231209032 and parameters: {'iterations': 150, 'learning_rate': 0.25, 'depth': 10, 'l2_leaf_reg': 1}. Best is trial 0 with value: 0.9530041503133905.
[CatBoost] Trial 2 starting...
[CatBoost] Trial 2 done, mean_balanced_acc=0.95802
[I 2026-04-27 16:25:20,760] Trial 2 finished with value: 0.9580166816417112 and parameters: {'iterations': 300, 'learning_rate': 0.15, 'depth': 6, 'l2_leaf_reg': 3}. Best is trial 2 with value: 0.9580166816417112.
[CatBoost] Trial 3 starting...
[CatBoost] Trial 3 done, mean_balanced_acc=0.95263
[I 2026-04-27 16:25:27,242] T

In [21]:
# Single stratified 80/20 train/test split on preprocessed *train*.
# Fit LightGBM + CatBoost on the train fold only; balanced accuracy on the held-out 20% test fold.
# Fitted estimators: models_fit["LightGBM"], models_fit["CatBoost"].
from sklearn.base import clone
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np

idx = np.arange(len(y_all_models))
tr_idx, va_idx = train_test_split(
    idx, test_size=0.2, stratify=y_all_models, random_state=42, shuffle=True
)

Xtr_g, Xva_g = X_all_models.iloc[tr_idx], X_all_models.iloc[va_idx]
ytr, yva = y_all_models.iloc[tr_idx], y_all_models.iloc[va_idx]

Xtr_c = X_catboost_raw.iloc[tr_idx].reset_index(drop=True)
Xva_c = X_catboost_raw.iloc[va_idx].reset_index(drop=True)
ytr_c = y_catboost_raw.iloc[tr_idx].reset_index(drop=True)

models_fit = {
    "LightGBM": clone(best_lgb),
    "CatBoost": clone(best_catboost),
}

print("\n=== 80/20 stratified train/test split ===\n", flush=True)
holdout_ba = {}
for name, m in models_fit.items():
    print(f"[{name}] Starting fit…", flush=True)
    if name == "CatBoost":
        m.set_params(verbose=100)
        m.fit(Xtr_c, ytr_c, cat_features=cat_features)
        pred = m.predict(Xva_c)
    else:
        m.set_params(verbosity=2)
        m.fit(Xtr_g, ytr)
        pred = m.predict(Xva_g)
    holdout_ba[name] = balanced_accuracy_score(yva, pred)
    print(
        f"[{name}] Done. Balanced accuracy (test fold, 20%): {holdout_ba[name]:.5f}\n",
        flush=True,
    )

metrics_compare = pd.DataFrame(
    {"model": list(holdout_ba.keys()), "balanced_accuracy": list(holdout_ba.values())}
).sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
best_ba = metrics_compare["balanced_accuracy"].max()
metrics_compare["gap_to_best"] = (best_ba - metrics_compare["balanced_accuracy"]).round(5)
metrics_compare["rank"] = range(1, len(metrics_compare) + 1)
metrics_compare = metrics_compare[["rank", "model", "balanced_accuracy", "gap_to_best"]]

metrics_compare


=== 80/20 stratified train/test split ===

[LightGBM] Starting fit…
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.000000
[LightGBM] [Debug] init for col-wise cost 0.000017 seconds, init for row-wise cost 0.072093 seconds
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.091741 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 19853
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 101
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 9
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 8
[LightGBM] [Debug] Traine

,rank,model,balanced_accuracy,gap_to_best
0,1,CatBoost,0.963464,0.00000
1,2,LightGBM,0.962790,0.00067


In [ ]:
# Native feature importance (LightGBM gain, CatBoost default) + permutation importance on a
# stratified sample of the *same 20% test fold* used above (models_fit must be fitted).
import numpy as np
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

TOP_K = 25
PERM_N = min(15_000, len(yva))
PERM_REPEATS = 5
perm_pos, _ = train_test_split(
    np.arange(len(yva)),
    train_size=PERM_N,
    stratify=yva,
    random_state=42,
)
X_perm_lgb = Xva_g.iloc[perm_pos]
X_perm_cb = Xva_c.iloc[perm_pos]
y_perm = yva.iloc[perm_pos]

lgb_fit = models_fit["LightGBM"]
cb_fit = models_fit["CatBoost"]

# --- LightGBM: gain-based importance ---
lgb_gain = lgb_fit.booster_.feature_importance(importance_type="gain")
lgb_imp = (
    pd.DataFrame({"feature": X_all_models.columns, "gain": lgb_gain})
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)
lgb_imp["rank_lgb_gain"] = range(1, len(lgb_imp) + 1)

# --- CatBoost ---
cb_imp_vals = cb_fit.get_feature_importance()
cb_imp = (
    pd.DataFrame({"feature": X_catboost_raw.columns, "importance": cb_imp_vals})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
cb_imp["rank_catboost"] = range(1, len(cb_imp) + 1)

# --- Permutation importance (balanced accuracy), separate per model ---
perm_lgb = permutation_importance(
    lgb_fit,
    X_perm_lgb,
    y_perm,
    scoring="balanced_accuracy",
    n_repeats=PERM_REPEATS,
    random_state=42,
    n_jobs=-1,
)
perm_lgb_df = (
    pd.DataFrame(
        {
            "feature": X_all_models.columns,
            "perm_mean": perm_lgb.importances_mean,
            "perm_std": perm_lgb.importances_std,
        }
    )
    .sort_values("perm_mean", ascending=False)
    .reset_index(drop=True)
)
perm_lgb_df["rank_perm_lgb"] = range(1, len(perm_lgb_df) + 1)

perm_cb = permutation_importance(
    cb_fit,
    X_perm_cb,
    y_perm,
    scoring="balanced_accuracy",
    n_repeats=PERM_REPEATS,
    random_state=42,
    n_jobs=-1,
)
perm_cb_df = (
    pd.DataFrame(
        {
            "feature": X_catboost_raw.columns,
            "perm_mean": perm_cb.importances_mean,
            "perm_std": perm_cb.importances_std,
        }
    )
    .sort_values("perm_mean", ascending=False)
    .reset_index(drop=True)
)
perm_cb_df["rank_perm_catboost"] = range(1, len(perm_cb_df) + 1)

print(f"Permutation importance: n={len(y_perm)} test rows (stratified), repeats={PERM_REPEATS}\n")

display(lgb_imp.head(TOP_K))
display(cb_imp.head(TOP_K))
display(perm_lgb_df.head(TOP_K))
display(perm_cb_df.head(TOP_K))

Permutation importance: n=15000 test rows (stratified), repeats=5



,feature,gain,rank_lgb_gain
0,Crop_Growth_Stage__target_High_te,759768.308481,1
1,Soil_Moisture,608112.420579,2
2,Temperature_C,399547.982829,3
3,Mulching_Used__target_High_te,394626.578105,4
4,Wind_Speed_kmh,244912.628296,5
5,Temperature_C Wind_Speed_kmh,169160.392104,6
6,Rainfall_mm,72861.410195,7
7,Soil_Moisture Rainfall_mm,57977.516619,8
8,Soil_pH Rainfall_mm,20255.749853,9
9,Temperature_C Rainfall_mm,14798.805702,10


,feature,importance,rank_catboost
0,Crop_Growth_Stage,28.468958,1
1,Soil_Moisture,27.837039,2
2,Mulching_Used,11.812726,3
3,Temperature_C,11.338198,4
4,Wind_Speed_kmh,10.835554,5
5,Rainfall_mm,3.448632,6
6,Humidity,2.200209,7
7,Previous_Irrigation_mm,1.317054,8
8,Sunlight_Hours,0.577137,9
9,Soil_pH,0.546905,10


,feature,perm_mean,perm_std,rank_perm_lgb
0,Soil_Moisture,0.333569,0.004088,1
1,Crop_Growth_Stage__target_High_te,0.282005,0.004199,2
2,Temperature_C,0.185239,0.006670,3
3,Mulching_Used__target_High_te,0.165515,0.006159,4
4,Wind_Speed_kmh,0.112705,0.003309,5
5,Rainfall_mm,0.040225,0.000923,6
6,Soil_Moisture Rainfall_mm,0.008095,0.001551,7
7,Temperature_C Rainfall_mm,0.004698,0.000760,8
8,Soil_pH Rainfall_mm,0.003192,0.000173,9
9,Rainfall_mm Sunlight_Hours,0.000638,0.000498,10


,feature,perm_mean,perm_std,rank_perm_catboost
0,Soil_Moisture,0.342875,0.005260,1
1,Crop_Growth_Stage,0.295969,0.003942,2
2,Temperature_C,0.198723,0.007066,3
3,Mulching_Used,0.185051,0.005454,4
4,Wind_Speed_kmh,0.173255,0.007032,5
5,Rainfall_mm,0.072100,0.001277,6
6,Season,0.000038,0.000000,7
7,Irrigation_Type,0.000015,0.000019,8
8,Crop_Type,-0.000008,0.000031,9
9,Soil_Type,-0.000012,0.000023,10


In [23]:
# LightGBM combined average rank: native gain + permutation rank
lgb_rank_avg = (
    lgb_imp[["feature", "rank_lgb_gain"]]
    .merge(perm_lgb_df[["feature", "rank_perm_lgb"]], on="feature", how="inner")
)
lgb_rank_avg["avg_rank_lgb"] = lgb_rank_avg[["rank_lgb_gain", "rank_perm_lgb"]].mean(axis=1)
lgb_rank_avg = lgb_rank_avg.sort_values("avg_rank_lgb", ascending=True).reset_index(drop=True)
lgb_rank_avg["rank_lgb_avg"] = range(1, len(lgb_rank_avg) + 1)
lgb_rank_avg = lgb_rank_avg[
    ["rank_lgb_avg", "feature", "rank_lgb_gain", "rank_perm_lgb", "avg_rank_lgb"]
]

print("LightGBM combined rank (native + permutation):")
display(lgb_rank_avg.head(TOP_K))

LightGBM combined rank (native + permutation):


,rank_lgb_avg,feature,rank_lgb_gain,rank_perm_lgb,avg_rank_lgb
0,1,Crop_Growth_Stage__target_High_te,1,2,1.5
1,2,Soil_Moisture,2,1,1.5
2,3,Temperature_C,3,3,3.0
3,4,Mulching_Used__target_High_te,4,4,4.0
4,5,Wind_Speed_kmh,5,5,5.0
5,6,Rainfall_mm,7,6,6.5
6,7,Soil_Moisture Rainfall_mm,8,7,7.5
7,8,Soil_pH Rainfall_mm,9,9,9.0
8,9,Temperature_C Rainfall_mm,10,8,9.0
9,10,Rainfall_mm Sunlight_Hours,14,10,12.0


In [24]:
# CatBoost combined average rank: native importance + permutation rank
cb_rank_avg = (
    cb_imp[["feature", "rank_catboost"]]
    .merge(perm_cb_df[["feature", "rank_perm_catboost"]], on="feature", how="inner")
)
cb_rank_avg["avg_rank_catboost"] = cb_rank_avg[["rank_catboost", "rank_perm_catboost"]].mean(axis=1)
cb_rank_avg = cb_rank_avg.sort_values("avg_rank_catboost", ascending=True).reset_index(drop=True)
cb_rank_avg["rank_catboost_avg"] = range(1, len(cb_rank_avg) + 1)
cb_rank_avg = cb_rank_avg[
    [
        "rank_catboost_avg",
        "feature",
        "rank_catboost",
        "rank_perm_catboost",
        "avg_rank_catboost",
    ]
]

print("CatBoost combined rank (native + permutation):")
display(cb_rank_avg.head(TOP_K))

CatBoost combined rank (native + permutation):


,rank_catboost_avg,feature,rank_catboost,rank_perm_catboost,avg_rank_catboost
0,1,Crop_Growth_Stage,1,2,1.5
1,2,Soil_Moisture,2,1,1.5
2,3,Mulching_Used,3,4,3.5
3,4,Temperature_C,4,3,3.5
4,5,Wind_Speed_kmh,5,5,5.0
5,6,Rainfall_mm,6,6,6.0
6,7,Sunlight_Hours,9,11,10.0
7,8,Previous_Irrigation_mm,8,14,11.0
8,9,Soil_pH,10,13,11.5
9,10,Humidity,7,16,11.5


In [25]:
# Compare all features vs top-50% features (based on average ranks) on the same 80/20 split
from sklearn.base import clone
from sklearn.metrics import balanced_accuracy_score

# --- Select top half features by average rank ---
lgb_top_n = max(1, len(lgb_rank_avg) // 2)
cb_top_n = max(1, len(cb_rank_avg) // 2)

lgb_top_features = lgb_rank_avg.nsmallest(lgb_top_n, "avg_rank_lgb")["feature"].tolist()
cb_top_features = cb_rank_avg.nsmallest(cb_top_n, "avg_rank_catboost")["feature"].tolist()

Xtr_g_half = Xtr_g[lgb_top_features]
Xva_g_half = Xva_g[lgb_top_features]
Xtr_c_half = Xtr_c[cb_top_features]
Xva_c_half = Xva_c[cb_top_features]

# Update cat_features indices for the reduced CatBoost matrix
cat_features_half = [i for i, c in enumerate(cb_top_features) if c in cat_cols]

results = []

# --- LightGBM: all features ---
lgb_all = clone(best_lgb)
lgb_all.fit(Xtr_g, ytr)
pred_lgb_all = lgb_all.predict(Xva_g)
ba_lgb_all = balanced_accuracy_score(yva, pred_lgb_all)
results.append(
    {
        "model": "LightGBM",
        "feature_set": "all",
        "n_features": Xtr_g.shape[1],
        "balanced_accuracy": ba_lgb_all,
    }
)

# --- LightGBM: top half features ---
lgb_half = clone(best_lgb)
lgb_half.fit(Xtr_g_half, ytr)
pred_lgb_half = lgb_half.predict(Xva_g_half)
ba_lgb_half = balanced_accuracy_score(yva, pred_lgb_half)
results.append(
    {
        "model": "LightGBM",
        "feature_set": "top_half_by_avg_rank",
        "n_features": Xtr_g_half.shape[1],
        "balanced_accuracy": ba_lgb_half,
    }
)

# --- CatBoost: all features ---
cb_all = clone(best_catboost)
cb_all.fit(Xtr_c, ytr_c, cat_features=cat_features)
pred_cb_all = cb_all.predict(Xva_c)
ba_cb_all = balanced_accuracy_score(yva, pred_cb_all)
results.append(
    {
        "model": "CatBoost",
        "feature_set": "all",
        "n_features": Xtr_c.shape[1],
        "balanced_accuracy": ba_cb_all,
    }
)

# --- CatBoost: top half features ---
cb_half = clone(best_catboost)
cb_half.fit(Xtr_c_half, ytr_c, cat_features=cat_features_half)
pred_cb_half = cb_half.predict(Xva_c_half)
ba_cb_half = balanced_accuracy_score(yva, pred_cb_half)
results.append(
    {
        "model": "CatBoost",
        "feature_set": "top_half_by_avg_rank",
        "n_features": Xtr_c_half.shape[1],
        "balanced_accuracy": ba_cb_half,
    }
)

half_feature_comparison = pd.DataFrame(results).sort_values(
    ["model", "feature_set"]
).reset_index(drop=True)

half_feature_comparison

,model,feature_set,n_features,balanced_accuracy
0,CatBoost,all,19,0.963464
1,CatBoost,top_half_by_avg_rank,9,0.963917
2,LightGBM,all,101,0.962790
3,LightGBM,top_half_by_avg_rank,50,0.963475


In [26]:
# LightGBM: remove 20 more features from the half set and compare
from sklearn.base import clone
from sklearn.metrics import balanced_accuracy_score

# Half set size from average-rank list
lgb_half_n = max(1, len(lgb_rank_avg) // 2)
lgb_half_features = lgb_rank_avg.nsmallest(lgb_half_n, "avg_rank_lgb")["feature"].tolist()

# Remove another 20 features from the half set (keep top-ranked ones)
lgb_minus20_n = max(1, len(lgb_half_features) - 20)
lgb_minus20_features = lgb_half_features[:lgb_minus20_n]

# Train/evaluate half-feature model
lgb_half_model = clone(best_lgb)
lgb_half_model.fit(Xtr_g[lgb_half_features], ytr)
pred_half = lgb_half_model.predict(Xva_g[lgb_half_features])
ba_half = balanced_accuracy_score(yva, pred_half)

# Train/evaluate half-minus-20 model
lgb_minus20_model = clone(best_lgb)
lgb_minus20_model.fit(Xtr_g[lgb_minus20_features], ytr)
pred_minus20 = lgb_minus20_model.predict(Xva_g[lgb_minus20_features])
ba_minus20 = balanced_accuracy_score(yva, pred_minus20)

lgb_minus20_comparison = pd.DataFrame(
    [
        {
            "model": "LightGBM",
            "feature_set": "top_half_by_avg_rank",
            "n_features": len(lgb_half_features),
            "balanced_accuracy": ba_half,
        },
        {
            "model": "LightGBM",
            "feature_set": "top_half_minus_20",
            "n_features": len(lgb_minus20_features),
            "balanced_accuracy": ba_minus20,
        },
    ]
)
lgb_minus20_comparison["delta_vs_half"] = (
    lgb_minus20_comparison["balanced_accuracy"] - ba_half
).round(6)

lgb_minus20_comparison

,model,feature_set,n_features,balanced_accuracy,delta_vs_half
0,LightGBM,top_half_by_avg_rank,50,0.963475,0.000000
1,LightGBM,top_half_minus_20,30,0.963229,-0.000246


In [27]:
# Stacking (meta-model) with half-feature LightGBM + half-feature CatBoost
# Compare balanced accuracy vs each half-feature base model on the same 20% holdout.
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd

# --- Rebuild half-feature sets from average-rank tables ---
lgb_half_n = max(1, len(lgb_rank_avg) // 2)
cb_half_n = max(1, len(cb_rank_avg) // 2)

lgb_half_features = lgb_rank_avg.nsmallest(lgb_half_n, "avg_rank_lgb")["feature"].tolist()
cb_half_features = cb_rank_avg.nsmallest(cb_half_n, "avg_rank_catboost")["feature"].tolist()

Xtr_g_half = Xtr_g[lgb_half_features]
Xva_g_half = Xva_g[lgb_half_features]
Xtr_c_half = Xtr_c[cb_half_features]
Xva_c_half = Xva_c[cb_half_features]
cat_features_half = [i for i, c in enumerate(cb_half_features) if c in cat_cols]

# --- Baselines: half-feature base models ---
lgb_half_model = clone(best_lgb)
lgb_half_model.fit(Xtr_g_half, ytr)
pred_lgb_half = lgb_half_model.predict(Xva_g_half)
ba_lgb_half = balanced_accuracy_score(yva, pred_lgb_half)

cb_half_model = clone(best_catboost)
cb_half_model.fit(Xtr_c_half, ytr_c, cat_features=cat_features_half)
pred_cb_half = cb_half_model.predict(Xva_c_half)
ba_cb_half = balanced_accuracy_score(yva, pred_cb_half)

# --- OOF stacking features for meta-model (train fold only) ---
classes = np.array(sorted(ytr.unique()))
class_to_idx = {c: i for i, c in enumerate(classes)}
n_classes = len(classes)

oof_lgb = np.zeros((len(ytr), n_classes), dtype=float)
oof_cb = np.zeros((len(ytr), n_classes), dtype=float)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for tr_fold, va_fold in skf.split(Xtr_g_half, ytr):
    # LightGBM fold
    lgb_fold = clone(best_lgb)
    lgb_fold.fit(Xtr_g_half.iloc[tr_fold], ytr.iloc[tr_fold])
    proba_lgb = lgb_fold.predict_proba(Xtr_g_half.iloc[va_fold])
    for j, cls in enumerate(lgb_fold.classes_):
        oof_lgb[va_fold, class_to_idx[cls]] = proba_lgb[:, j]

    # CatBoost fold
    cb_fold = clone(best_catboost)
    cb_fold.fit(
        Xtr_c_half.iloc[tr_fold],
        ytr_c.iloc[tr_fold],
        cat_features=cat_features_half,
    )
    proba_cb = cb_fold.predict_proba(Xtr_c_half.iloc[va_fold])
    for j, cls in enumerate(cb_fold.classes_):
        oof_cb[va_fold, class_to_idx[cls]] = proba_cb[:, j]

X_meta_train = np.hstack([oof_lgb, oof_cb])

meta_model = LogisticRegression(max_iter=3000, random_state=42)
meta_model.fit(X_meta_train, ytr)

# --- Holdout meta features from base models fit on full train split ---
lgb_full_for_meta = clone(best_lgb)
cb_full_for_meta = clone(best_catboost)

lgb_full_for_meta.fit(Xtr_g_half, ytr)
cb_full_for_meta.fit(Xtr_c_half, ytr_c, cat_features=cat_features_half)

proba_lgb_holdout = lgb_full_for_meta.predict_proba(Xva_g_half)
proba_cb_holdout = cb_full_for_meta.predict_proba(Xva_c_half)

# Align holdout probability columns to the same class order used in meta training
lgb_holdout_aligned = np.zeros((len(yva), n_classes), dtype=float)
cb_holdout_aligned = np.zeros((len(yva), n_classes), dtype=float)

for j, cls in enumerate(lgb_full_for_meta.classes_):
    lgb_holdout_aligned[:, class_to_idx[cls]] = proba_lgb_holdout[:, j]
for j, cls in enumerate(cb_full_for_meta.classes_):
    cb_holdout_aligned[:, class_to_idx[cls]] = proba_cb_holdout[:, j]

X_meta_holdout = np.hstack([lgb_holdout_aligned, cb_holdout_aligned])
pred_stack = meta_model.predict(X_meta_holdout)
ba_stack = balanced_accuracy_score(yva, pred_stack)

stack_compare = pd.DataFrame(
    [
        {
            "model": "LightGBM_half_features",
            "n_features": len(lgb_half_features),
            "balanced_accuracy": ba_lgb_half,
        },
        {
            "model": "CatBoost_half_features",
            "n_features": len(cb_half_features),
            "balanced_accuracy": ba_cb_half,
        },
        {
            "model": "Stacking_meta_LGBM+CatBoost",
            "n_features": f"{len(lgb_half_features)} + {len(cb_half_features)} (meta probs)",
            "balanced_accuracy": ba_stack,
        },
    ]
).sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)

stack_compare

,model,n_features,balanced_accuracy
0,Stacking_meta_LGBM+CatBoost,50 + 9 (meta probs),0.963999
1,CatBoost_half_features,9,0.963917
2,LightGBM_half_features,50,0.963475


In [28]:

from pathlib import Path
import numpy as np
import pandas as pd

# --- Load test + sample submission ---
test_raw = pd.read_csv("playground-series-s6e4/test.csv")
sample_sub = pd.read_csv("playground-series-s6e4/sample_submission.csv")

# Keep id for final submission, drop from feature frame
ids_test = test_raw["id"].copy()
test_features = test_raw.drop(columns=["id"]).copy()

# --- Recreate LightGBM feature matrix for test ---
X_test_poly = pd.DataFrame(
    poly.transform(test_features[num_cols]),
    columns=poly.get_feature_names_out(num_cols),
    index=test_features.index,
)

# Full-train target encoding stats per categorical feature
# (inference-safe fallback to class prior for unseen categories)
class_prior = y_onehot.mean()
X_test_cat_encoded = pd.DataFrame(index=test_features.index)

for col in cat_cols:
    stats = y_onehot.groupby(train[col]).mean()
    mapped = test_features[col].map(stats.to_dict("index"))
    for cls in class_cols:
        X_test_cat_encoded[f"{col}__{cls}_te"] = [
            d.get(cls, class_prior[cls]) if isinstance(d, dict) else class_prior[cls]
            for d in mapped
        ]

X_test_all_models = pd.concat([X_test_poly, X_test_cat_encoded], axis=1)

# Align to the feature subsets used by the stacked base models
X_test_lgb_half = X_test_all_models[lgb_half_features]
X_test_cb_half = test_features[cb_half_features].copy()

# --- Stacked prediction ---
proba_lgb_test = lgb_full_for_meta.predict_proba(X_test_lgb_half)
proba_cb_test = cb_full_for_meta.predict_proba(X_test_cb_half)

n_classes = len(class_to_idx)
lgb_test_aligned = np.zeros((len(X_test_lgb_half), n_classes), dtype=float)
cb_test_aligned = np.zeros((len(X_test_cb_half), n_classes), dtype=float)

for j, cls in enumerate(lgb_full_for_meta.classes_):
    lgb_test_aligned[:, class_to_idx[cls]] = proba_lgb_test[:, j]
for j, cls in enumerate(cb_full_for_meta.classes_):
    cb_test_aligned[:, class_to_idx[cls]] = proba_cb_test[:, j]

X_meta_test = np.hstack([lgb_test_aligned, cb_test_aligned])
test_pred = meta_model.predict(X_meta_test)

# --- Write submission to Output folder ---
out_dir = Path("Output")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "submission_stacked.csv"

submission = sample_sub.copy()
submission["id"] = ids_test.values
submission["Irrigation_Need"] = test_pred
submission.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
submission.head()

Saved: Output/submission_stacked.csv


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
